# Agent Planning

이 노트북은 planning 주제를 안전한 추가 학습 레이어로 다룬다. main workflow를 바꾸지 않고, planning이 왜 중요한지, ReAct와 planner-executor가 어떻게 다른지, task decomposition이 agent behavior를 왜 더 읽기 쉽게 만드는지를 설명한다.

## 학습 목표

- agent planning이 왜 중요한지 이해한다.
- ReAct와 planner-executor 패턴을 비교한다.
- task decomposition이 plan을 어떻게 바꾸는지 본다.
- explicit handler로 단순한 plan을 직접 실행해본다.


## 개념 설명

다른 노트북과 마찬가지로 먼저 interpreter를 확인한다. setup script가 등록한 `uv` 환경 안에서 이 notebook이 실행되고 있는지 확인하는 단계다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

이 setup 셀은 `src/planner_extended.py`에 있는 추가 planning helper를 불러온다. 기존 `src/planner.py`는 그대로 두고, 노트북에서는 tutorial 중심으로 planning 개념을 설명한다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.planner_extended import PlanExecutor, PlanGenerator

pd.set_option('display.max_colwidth', 140)
generator = PlanGenerator()


planning은 agent가 모호한 요청을 더 작은 실행 단위로 나누게 해준다. plan이 없으면 agent는 곧바로 synthesis로 뛰어들기 쉽고, 그러면 어떤 단계에서 실수가 생겼는지 디버깅하기가 어려워진다.

ReAct는 observe, think, act, observe 루프에 가깝고, planner-executor는 계획과 실행을 분리한다. task decomposition은 넓은 요청을 더 직접적인 sub-task로 나누는 데 초점을 둔다.

## 구현


In [ ]:
planning_task = 'Find the rollout date, compare it to the pilot window, and summarize why the timing matters.'
strategy_comparison = generator.compare_strategies(planning_task)
{
    strategy: len(plan)
    for strategy, plan in strategy_comparison.items()
}


아래 표는 planning 스타일 차이를 눈에 보이게 만든다. ReAct는 reasoning 지향 step을, planner-executor는 깔끔한 실행 outline을, decomposition은 더 직접적인 하위 문제 분해를 보여준다.


In [ ]:
plan_frames = {
    strategy: pd.DataFrame(plan)
    for strategy, plan in strategy_comparison.items()
}
plan_frames['react'], plan_frames['planner_executor'], plan_frames['decompose']


planner는 executor와 연결될 때 훨씬 유용해진다. 아래 executor는 explicit handler를 사용해서, 어떤 step이 retrieval을 요청하는지, 어떤 step이 tool을 요청하는지 노트북에서 그대로 볼 수 있게 한다.


In [ ]:
executor = PlanExecutor()
executor.register_handler('retrieval', lambda step: {'status': 'completed', 'output': f"Retrieved evidence for: {step['objective']}"})
executor.register_handler('tool', lambda step: {'status': 'completed', 'output': f"Ran a focused tool for: {step['objective']}"})
planner_executor_plan = generator.generate_plan(planning_task, strategy='planner_executor')
execution_log = executor.execute(planner_executor_plan)
pd.DataFrame(execution_log)


## 실험

좋은 planning 실험은 task가 단순할 때와 multi-step일 때 plan이 어떻게 달라지는지 비교하는 것이다. 이 셀은 여러 요청에 대해 plan을 생성해서 decomposition이 task 구조에 어떻게 반응하는지 보여준다.


In [ ]:
experiment_tasks = [
    'Summarize the rollout plan.',
    'Find the rollout date and explain who needs to know it.',
    'Search the policy goals, calculate the pilot duration, and draft a short update.',
]
experiment_rows = []
for task in experiment_tasks:
    plan = generator.generate_plan(task, strategy='decompose')
    experiment_rows.append({'task': task, 'plan_length': len(plan), 'first_step': plan[0]['objective']})
pd.DataFrame(experiment_rows)


## 결과 해석

planning 결과를 보면, 좋은 agent에 하나의 만능 planning 방식이 필요한 것은 아니라는 점이 드러난다. ReAct는 반복적 reasoning에, planner-executor는 계획과 실행 경계를 깔끔하게 나눌 때, decomposition은 sub-part가 분명한 문제에 특히 잘 맞는다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'strategy': 'react', 'strength': 'good for iterative observe-think-act loops'},
        {'strategy': 'planner_executor', 'strength': 'good for explicit planning and safe execution'},
        {'strategy': 'decompose', 'strength': 'good for turning broad tasks into manageable chunks'},
    ]
)
analysis_frame


## 핵심 정리

- 이 실험을 통해 planning은 agent behavior를 더 잘 보이게 만들고, 디버깅 가능성도 높인다는 점을 확인했다.
- ReAct, planner-executor, task decomposition은 서로 다른 목적을 가진다.
- explicit execution log가 있으면 planning 품질과 tool 품질을 분리해서 볼 수 있다.
- 면접에서는 "planning을 왜 넣었는가"라는 질문에 대해, **reasoning을 숨기기 위해서가 아니라 workflow를 더 explainable하게 만들기 위해서**라고 답할 수 있다.
